# Feedzai Base: actual author-cell outputs

100-trial CPU replay with documented compatibility changes.

In [1]:
import sys
import os
import json
import time
import importlib.metadata
from pathlib import Path
sys.dont_write_bytecode = True
sys.path.insert(0, 'D:\\UIT\\DS317\\BAF_project\\external\\feedzai08_runtime\\packages')
sys.path.insert(0, 'D:\\UIT\\DS317\\BAF_project\\results\\runs\\feedzai_base_original_20261028_003248_73856e42\\evidence')
import numpy as np
np.random.seed(42)
DATA_PATH = 'D:\\UIT\\DS317\\BAF_project\\data\\raw\\Base.csv'
N_TRIALS = 100
N_JOBS = 10
EXPECTED_ROWS = 1000000
exec(Path('D:/UIT/DS317/BAF_project/results/runs/feedzai_base_original_20261028_003248_73856e42/evidence/capture_results.py').read_text(encoding='utf-8'), globals())

## Author cell 0

In [2]:
# coding=utf-8
#
# The copyright of this file belongs to Feedzai. The file cannot be
# reproduced in whole or in part, stored in a retrieval system,
# transmitted in any form, or by any means electronic, mechanical,
# photocopying, or otherwise, without the prior permission of the owner.
#
# (c) 2022 Feedzai, Strictly Confidential

## Author cell 1

In [3]:
import lightgbm as lgbm  # Tested ML method
import numpy as np       # Random number generation
import seaborn as sns    # Plotting library
import pandas as pd      # Read/write data
import yaml              # Read hyperparameter space configuration

# Group is provided by the documented Aequitas 0.42 adapter.                # Fairness metrics
from matplotlib import pyplot as plt            # Plotting method
from sklearn.preprocessing import LabelEncoder  # Categorical encoding for LGBM models
from sklearn import metrics                     # ROC metrics

from random_search import RandomValueTrial, suggest_callable_hyperparams  # Random search wrapper methods
lgbm.register_logger(QuietLightGBM())


## Author cell 2

In [4]:
# Read hyperparameter space for the LGBM Models, expected structure is presented bellow
with open("lightgbm_hyperparameter_space.yaml", "r") as file:
    hyperparam_space = yaml.load(file, Loader=yaml.FullLoader)

## Author cell 3

In [5]:
# The expected structure is the following:
hyperparam_space

Out[5]: 
{'LightGBM': {'classpath': 'lightgbm.LGBMClassifier',
  'kwargs': {'n_estimators': {'type': 'int',
    'range': [20, 10000],
    'log': True},
   'max_depth': {'type': 'int', 'range': [3, 30]},
   'learning_rate': {'type': 'float', 'range': [0.02, 0.1], 'log': True},
   'num_leaves': {'type': 'int', 'range': [10, 100], 'log': True},
   'boosting_type': ['gbdt', 'goss'],
   'min_data_in_leaf': {'type': 'int', 'range': [5, 200], 'log': True},
   'max_bin': {'type': 'int', 'range': [100, 500]},
   'enable_bundle': [True, False]}}}


## Author cell 4

In [6]:
# Testing a random search suggestion:
trial = RandomValueTrial(seed=1)
suggest_callable_hyperparams(trial, hyperparam_space)

Out[6]: 
{'classpath': 'lightgbm.LGBMClassifier',
 'n_estimators': 263,
 'max_depth': 23,
 'learning_rate': 0.020003681922217444,
 'num_leaves': 19,
 'boosting_type': 'gbdt',
 'min_data_in_leaf': 9,
 'max_bin': 238,
 'enable_bundle': False}


## Author cell 5

In [7]:
datasets_paths = {"Base": str(DATA_PATH)}


## Author cell 6

In [8]:
# Read the datasets with pandas.
datasets = {key: pd.read_csv(path) for key, path in datasets_paths.items()}
assert len(datasets['Base']) == EXPECTED_ROWS
assert set(datasets['Base'].fraud_bool.unique()) == {0, 1}


## Author cell 7

In [9]:
# Define the label field and categorical columns.
label = "fraud_bool"

categorical_features = [
    "payment_type",
    "employment_status",
    "housing_status",
    "source",
    "device_os",
]

## Author cell 8

In [10]:
# Create the train and test sets. Shuffle data with `sample` method.
# The split was done by month. The first 6 months as the train, the last 2 months as test.
train_dfs = {key: df[df["month"]<6].sample(frac=1, replace=False) for key, df in datasets.items()}
test_dfs = {key: df[df["month"]>=6].sample(frac=1, replace=False) for key, df in datasets.items()}
save_split_evidence()


## Author cell 9

In [11]:
# Encode the categorical variables in the datasets to integers. 
# This is expected by LGBM (or columns with the `categorical` data type).

for name in datasets.keys():  # For each dataset in the suite
    train = train_dfs[name]
    test = test_dfs[name]

    for feat in categorical_features:
        encoder = LabelEncoder()
        encoder.fit(train[feat])  # Fit an encoder to the train set.
        train[feat] = encoder.transform(train[feat])  # Transform train set.
        test[feat] = encoder.transform(test[feat])    # Transform test set.

## Author cell 10

In [12]:
# Cell with train loop.

# Define number of trials in Random search.
n_trials=N_TRIALS
# Random state for sampling seeds.
np.random.seed(42)
# Seeds for the random search sampling algorithm.
seeds = np.random.choice(list(range(1_000_000)), size=n_trials, replace=False)

# Variable to store the results.
runs = {}

for trial in range(n_trials):
    seed = seeds[trial]
    trial = RandomValueTrial(seed=seed)
    # Hyperparameters for the random search trial.
    test_hyperparams = suggest_callable_hyperparams(trial, hyperparam_space)
    del test_hyperparams["classpath"] # Remove unnecessary key in hyperparaemters.
    
    # Update list of tested hyperparameters.
    prev_hyperparams = runs.get("hyperparams", [])
    prev_hyperparams.append(test_hyperparams)
    runs["hyperparams"] = prev_hyperparams
    
    for dataset_name in datasets.keys():  # Run hyperparameters on all variants of datastes.
        model = lgbm.LGBMClassifier(n_jobs=N_JOBS, **test_hyperparams)  # Instantiate LGBM Model.
        X_train = train_dfs[dataset_name].drop(columns=["fraud_bool"])
        y_train = train_dfs[dataset_name]["fraud_bool"]
        X_test = test_dfs[dataset_name].drop(columns=["fraud_bool"])
        y_test = test_dfs[dataset_name]["fraud_bool"]
        # Fit model to training data.
        print("FIT_TRIAL", len(runs.get(dataset_name, [])) + 1, test_hyperparams, flush=True)
        fit_started = time.perf_counter()
        model.fit(X_train, y_train, categorical_feature=categorical_features)
        fit_seconds = time.perf_counter() - fit_started
        # Obtain predictions in test data.
        predictions = model.predict_proba(X_test)[:, 1]
        
        # Obtain ROC curve for the predictions.
        fprs, tprs, thresholds = metrics.roc_curve(y_test, predictions)
        # Obtain threshold and recall. We select 5% FPR as threshold.
        threshold = np.min(thresholds[fprs==max(fprs[fprs < 0.05])])
        recall = np.max(tprs[fprs==max(fprs[fprs < 0.05])])

        # Binarize predictions for Aequitas.
        preds_binary = (predictions > threshold).astype(int)
        
        # Create a dataframe with protected group column, predictions and labels.
        # Here, we select age>=50 as threshold.
        aequitas_df = pd.DataFrame(
            {
                "age": (X_test["customer_age"]>=50).map({True: "Older", False: "Younger"}),
                "preds": preds_binary,
                "y": y_test.values
            }
        )
        
        # Obtain FPR results for different groups.
        g = Group()
        aequitas_results = g.get_crosstabs(aequitas_df, attr_cols=["age"], score_col="preds", label_col="y")[0]
        
        # Store the results for the trained model
        results = {}
        results["recall"] = recall
        results["recall Older"] = aequitas_results[aequitas_results["attribute_value"] == "Older"][["tpr"]].values[0][0]
        results["recall Younger"] = aequitas_results[aequitas_results["attribute_value"] == "Younger"][["tpr"]].values[0][0]
        results["fpr Older"] = aequitas_results[aequitas_results["attribute_value"] == "Older"][["fpr"]].values[0][0]
        results["fpr Younger"] = aequitas_results[aequitas_results["attribute_value"] == "Younger"][["fpr"]].values[0][0]
        
        # Store the results in the runs variable
        prev_runs = runs.get(dataset_name, [])
        prev_runs.append(results)
        runs[dataset_name] = prev_runs
        capture_trial(len(prev_runs), int(seed), model, predictions, threshold, recall, results)


FIT_TRIAL 1 {'n_estimators': 282, 'max_depth': 5, 'learning_rate': 0.08045625153147128, 'num_leaves': 65, 'boosting_type': 'goss', 'min_data_in_leaf': 22, 'max_bin': 220, 'enable_bundle': True}
[LightGBM] [Warning] min_data_in_leaf is set=22, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=22
[LightGBM] [Warning] Found boosting=goss. For backwards compatibility reasons, LightGBM interprets this as boosting=gbdt, data_sample_strategy=goss.To suppress this warning, set data_sample_strategy=goss instead.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
FEEDZAI_TRIAL 1/100 recall=0.530577 AP=0.181734 fit=12.9s
FIT_TRIAL 2 {'n_estimators': 20, 'max_depth': 4, 'learning_rate': 0.07306386606820443, 'num_leaves': 11, 'boosting_type': 'goss', 'min_data_in_leaf': 9, 'max_bin': 254, 'enable_bundle': False}
[LightGBM] [Warning] min_data_in_leaf is set=9, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=9
FEEDZAI_TRIAL 2/100 recal

## Author cell 11

In [13]:
# Create a dataframe with the results for each model in each dataset.
rs_results = pd.DataFrame(runs)

## Author cell 12

In [14]:
# Helper method to obtain the metric values for a given model.
def get_results(results, variant, metric):
    col = results[variant]
    values = []
    for idx, val in col.items():
        values.append(val[metric])
    return values

## Author cell 13

In [15]:
# Obtain the relevant metrics to plots from the dataframe.
variants = list(datasets_paths.keys())

plot_results = {"Variant": [], "Recall": [], "FPR Ratio": []}

for variant in variants:
    plot_results["Recall"] += get_results(rs_results, variant, "recall")
    # Obtain the FPR if both groups.
    for fpr_younger, fpr_older in zip(get_results(rs_results, variant, "fpr Younger"), get_results(rs_results, variant, "fpr Older")):
        # Calculate FPR ratio as higher fpr / lower fpr
        if fpr_younger > fpr_older:
            plot_results["FPR Ratio"] += [fpr_older/fpr_younger]
        else:
            plot_results["FPR Ratio"] += [fpr_younger/fpr_older]
    plot_results["Variant"] += [variant] * len(get_results(rs_results, variant, "recall"))

# Create a dataframe for easier plots.
plot_results = pd.DataFrame(plot_results)

In [16]:
finalize_results()

FEEDZAI_BASE_RESULTS_READY
    trial  source_roc_recall  average_precision  fit_seconds
83     84           0.557331           0.200558    22.058400
78     79           0.554899           0.198214    30.459648
20     21           0.552467           0.193422     5.443224
14     15           0.551772           0.199609    11.706496
62     63           0.550730           0.178531     8.795873
